# Notebook 04 v2 — SHAP Analysis (PyTorch DNN support)

**Purpose:** Compute SHAP values for all 18 5-class models (3 datasets × 6 model variants).

**Why hybrid-calibrated framing (Stance C):** Downstream analyses use hybrid-calibrated probabilities. SHAP values computed on raw model probabilities answer 'what features drive the model predictions which are then calibrated downstream?' — the same question Krishna agreement (Notebook 06) and SCTS-v2 (Notebooks 07-08) need.

**Methodology:**
- TreeExplainer for RF and XGBoost (exact SHAP, fast)
- GradientExplainer for PyTorch DNN with softmax wrapper (gives SHAP on probabilities)
- Background: 100 stratified samples from calibration set
- Evaluation: 1000 stratified samples from test set
- Output shape per model: (n_eval_samples, n_features, n_classes) → saved as .npy

**Overnight resilience:**
- Each model saved immediately after compute
- Resumable: re-running skips already-completed models
- Drive heartbeat check before each model
- Errors caught per model — one failure doesn't kill the whole run
- Final summary report at end

**v2 changes from v1:**
- DNN loading switched from Keras (.h5) to PyTorch (.pt) checkpoints
- DNN class hardcoded matching the observed checkpoint architecture (Linear→BN→ReLU→Dropout × 4 → Linear)
- GradientExplainer replaces DeepExplainer (more stable for PyTorch in recent SHAP versions)
- File extension auto-detection (RF/XGB are .pkl on NSL, .joblib on UNSW/CIC)

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import subprocess
try:
    import shap
    print(f'✓ SHAP {shap.__version__} already installed')
except ImportError:
    subprocess.check_call(['pip', 'install', '-q', 'shap'])
    import shap
    print(f'✓ Installed SHAP {shap.__version__}')

✓ SHAP 0.51.0 already installed


In [3]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import traceback
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import shap

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('  ⚠ Running on CPU — DNN SHAP will be slower')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

N_BACKGROUND = 100
N_EVAL = 1000

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']

MODELS_5CLASS = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
                 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']

print(f'\nConfig: B={N_BACKGROUND} background, N={N_EVAL} eval samples')
print(f'Total models to process: {len(DATASETS) * len(MODELS_5CLASS)}')

PyTorch device: cuda
  GPU: Tesla T4

Config: B=100 background, N=1000 eval samples
Total models to process: 18


## 2. DNN architecture (reconstructed from checkpoint inspection)

In [4]:
class DNN(nn.Module):
    """Architecture inferred from saved .pt checkpoint state_dict pattern:
    net.0=Linear, net.1=BatchNorm1d, net.2=ReLU, net.3=Dropout (repeats 4 times), net.16=Linear (output).
    Output is LOGITS — softmax applied in wrapper for SHAP.
    """
    def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class DNNWithSoftmax(nn.Module):
    """Wrapper that applies softmax to logits, for SHAP on probabilities."""
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
    def forward(self, x):
        logits = self.base(x)
        return torch.softmax(logits, dim=-1)

def load_pytorch_dnn(path):
    """Load DNN from checkpoint dict, return softmax-wrapped model in eval mode on DEVICE."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model = DNN(in_dim=ckpt['in_dim'], n_classes=ckpt['n_classes'],
                hidden=tuple(ckpt['hidden']), dropout=ckpt['dropout'])
    model.load_state_dict(ckpt['state_dict'])
    model = model.to(DEVICE)
    model.eval()  # disables dropout, freezes BN running stats
    wrapped = DNNWithSoftmax(model).to(DEVICE)
    wrapped.eval()
    return wrapped

# Quick sanity test on one DNN
test_path = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'dnn_5class_smote.pt'
if test_path.exists():
    test_model = load_pytorch_dnn(test_path)
    test_input = torch.randn(2, 122, device=DEVICE)
    with torch.no_grad():
        out = test_model(test_input)
    print(f'✓ DNN load test passed: output shape={tuple(out.shape)}, sums per row={out.sum(dim=1).cpu().numpy()}')
    print(f'  (sums should be ~1.0 since softmax applied)')
else:
    print(f'✗ Test model not found at {test_path}')

✓ DNN load test passed: output shape=(2, 5), sums per row=[1.0000001 1.       ]
  (sums should be ~1.0 since softmax applied)


## 3. Helper functions

In [5]:
def drive_alive():
    test = Path('/content/drive/MyDrive/XIDS_Research/xids-research/notebooks')
    if test.exists():
        return True
    print('⚠ Drive disconnected, attempting remount...')
    try:
        drive.mount('/content/drive', force_remount=True)
        return test.exists()
    except Exception as e:
        print(f'  Remount failed: {e}')
        return False

def stratified_sample(X, y, n, rng_local):
    n_per_class = max(1, n // len(np.unique(y)))
    idx_list = []
    for c in np.unique(y):
        class_idx = np.where(y == c)[0]
        take = min(n_per_class, len(class_idx))
        idx_list.append(rng_local.choice(class_idx, size=take, replace=False))
    idx = np.concatenate(idx_list)
    if len(idx) < n:
        remaining = list(set(range(len(y))) - set(idx.tolist()))
        extra = rng_local.choice(remaining, size=min(n - len(idx), len(remaining)), replace=False)
        idx = np.concatenate([idx, extra])
    return np.sort(idx[:n])

def find_tree_model_path(dataset, model_name):
    """RF/XGB models are .pkl on NSL, .joblib on UNSW/CIC."""
    base = Path(REPO) / 'models' / dataset
    for ext in ['.pkl', '.joblib']:
        p = base / f'{model_name}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'No tree model file for {model_name} in {base}')

def find_dnn_path(dataset, model_name):
    p = Path(REPO) / 'models' / dataset / f'{model_name}.pt'
    if p.exists():
        return p
    raise FileNotFoundError(f'No DNN file at {p}')

def already_done(dataset, model_name):
    out_dir = Path(REPO) / 'shap_values' / dataset
    return (out_dir / f'{model_name}_shap.npy').exists() and (out_dir / f'{model_name}_meta.json').exists()

def save_shap_outputs(dataset, model_name, shap_values, X_eval_idx, meta):
    out_dir = Path(REPO) / 'shap_values' / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f'{model_name}_shap.npy', shap_values)
    np.save(out_dir / f'{model_name}_eval_idx.npy', X_eval_idx)
    with open(out_dir / f'{model_name}_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

print('✓ Helpers loaded')

✓ Helpers loaded


## 4. SHAP computation per model

In [6]:
def compute_shap_for_model(dataset, model_name):
    """Compute SHAP values for one (dataset, model). Returns (meta, shape, elapsed)."""
    t0 = time.time()

    PROCESSED = Path(REPO) / 'data' / 'processed' / dataset
    X_test = np.load(PROCESSED / 'X_test.npy').astype(np.float32)
    y_test = np.load(PROCESSED / 'y_test_5class.npy')
    X_calib = np.load(PROCESSED / 'X_calib.npy').astype(np.float32)
    y_calib = np.load(PROCESSED / 'y_calib_5class.npy')

    n_classes = 5
    n_features = X_test.shape[1]

    rng_local = np.random.default_rng(SEED + abs(hash(model_name)) % 1000)
    bg_idx = stratified_sample(X_calib, y_calib, N_BACKGROUND, rng_local)
    eval_idx = stratified_sample(X_test, y_test, N_EVAL, rng_local)

    X_bg = X_calib[bg_idx]
    X_eval = X_test[eval_idx]

    model_type = 'rf' if 'rf' in model_name else ('xgb' if 'xgb' in model_name else 'dnn')

    if model_type in ('rf', 'xgb'):
        path = find_tree_model_path(dataset, model_name)
        model = joblib.load(path)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_eval, check_additivity=False)

        # Standardize to (n_eval, n_features, n_classes)
        if isinstance(shap_values, list):
            shap_values = np.stack(shap_values, axis=-1)
        elif shap_values.ndim == 3:
            # Some shap versions return (n_classes, n_samples, n_features)
            if shap_values.shape[0] == n_classes:
                shap_values = np.transpose(shap_values, (1, 2, 0))
            elif shap_values.shape[1] == n_classes:
                shap_values = np.transpose(shap_values, (0, 2, 1))

    elif model_type == 'dnn':
        path = find_dnn_path(dataset, model_name)
        wrapped_model = load_pytorch_dnn(path)

        # Convert to tensors
        bg_tensor = torch.from_numpy(X_bg).to(DEVICE)
        eval_tensor = torch.from_numpy(X_eval).to(DEVICE)

        # GradientExplainer is more stable than DeepExplainer for PyTorch
        explainer = shap.GradientExplainer(wrapped_model, bg_tensor)
        shap_values = explainer.shap_values(eval_tensor)

        # Standardize shape
        if isinstance(shap_values, list):
            shap_values = np.stack(shap_values, axis=-1)
        elif shap_values.ndim == 3:
            if shap_values.shape[0] == n_classes:
                shap_values = np.transpose(shap_values, (1, 2, 0))
            elif shap_values.shape[1] == n_classes:
                shap_values = np.transpose(shap_values, (0, 2, 1))

        # Cleanup GPU memory after DNN
        del wrapped_model, bg_tensor, eval_tensor
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    expected_shape = (N_EVAL, n_features, n_classes)
    if shap_values.shape != expected_shape:
        # Last-ditch shape fixes
        if shap_values.shape == (n_classes, N_EVAL, n_features):
            shap_values = np.transpose(shap_values, (1, 2, 0))
        elif shap_values.shape == (N_EVAL, n_classes, n_features):
            shap_values = np.transpose(shap_values, (0, 2, 1))

    t_elapsed = time.time() - t0

    meta = {
        'dataset': dataset,
        'model': model_name,
        'model_type': model_type,
        'shap_shape': list(shap_values.shape),
        'n_background': len(bg_idx),
        'n_eval': len(eval_idx),
        'n_features': n_features,
        'n_classes': n_classes,
        'time_seconds': round(t_elapsed, 1),
        'timestamp': datetime.now().isoformat(),
        'explainer': 'TreeExplainer' if model_type in ('rf', 'xgb') else 'GradientExplainer',
        'shap_target': 'probabilities (softmax)' if model_type == 'dnn' else 'model output (tree)',
        'note': 'SHAP computed on raw model output. For DNN, softmax wrapper applied so SHAP is on probabilities. Calibration is a post-hoc transformation applied separately; SHAP shows what features drive the underlying model predictions which then get calibrated downstream.'
    }

    save_shap_outputs(dataset, model_name, shap_values, eval_idx, meta)
    return meta, shap_values.shape, t_elapsed

## 5. Main loop

In [ ]:
results_log = []
errors_log = []
t_overall = time.time()

print(f'\n{"="*70}')
print(f'SHAP computation starting at {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'{"="*70}\n')

for dataset in DATASETS:
    print(f'\n{"="*70}')
    print(f'Dataset: {dataset}')
    print('='*70)

    for model_name in MODELS_5CLASS:
        if not drive_alive():
            print(f'❌ Drive dead before {dataset}/{model_name}. Stopping.')
            errors_log.append({'dataset': dataset, 'model': model_name, 'error': 'Drive disconnect'})
            break

        if already_done(dataset, model_name):
            print(f'  ⏭  {model_name:<22} (already computed, skipping)')
            results_log.append({'dataset': dataset, 'model': model_name, 'status': 'skipped'})
            continue

        print(f'  ▶  {model_name:<22} ', end='', flush=True)
        try:
            meta, shape, elapsed = compute_shap_for_model(dataset, model_name)
            print(f'shape={shape}, {elapsed:.1f}s')
            results_log.append({'dataset': dataset, 'model': model_name, 'status': 'done', 'time_s': elapsed, 'shape': str(shape)})
        except Exception as e:
            print(f'❌ FAILED: {type(e).__name__}: {e}')
            errors_log.append({
                'dataset': dataset, 'model': model_name,
                'error': f'{type(e).__name__}: {e}',
                'traceback': traceback.format_exc()
            })
            continue

t_total = time.time() - t_overall
print(f'\n{"="*70}')
print(f'Complete in {t_total/60:.1f} minutes at {datetime.now().strftime("%H:%M:%S")}')
print(f'  Successful: {sum(1 for r in results_log if r["status"] == "done")}')
print(f'  Skipped: {sum(1 for r in results_log if r["status"] == "skipped")}')
print(f'  Failed: {len(errors_log)}')
print('='*70)


SHAP computation starting at 2026-05-29 05:00:08


Dataset: nsl_kdd_v2
  ▶  rf_5class_smote        shape=(1000, 122, 5), 167.3s
  ▶  xgb_5class_smote       shape=(1000, 122, 5), 13.8s
  ▶  dnn_5class_smote       shape=(1000, 122, 5), 326.1s
  ▶  rf_5class_cw           shape=(1000, 122, 5), 133.0s
  ▶  xgb_5class_cw          shape=(1000, 122, 5), 10.3s
  ▶  dnn_5class_cw          shape=(1000, 122, 5), 334.2s

Dataset: unsw_nb15_v2
  ▶  rf_5class_smote        shape=(1000, 194, 5), 1621.2s
  ▶  xgb_5class_smote       shape=(1000, 194, 5), 2.9s
  ▶  dnn_5class_smote       ❌ FAILED: KeyError: 'in_dim'
  ▶  rf_5class_cw           shape=(1000, 194, 5), 1278.0s
  ▶  xgb_5class_cw          shape=(1000, 194, 5), 2.1s
  ▶  dnn_5class_cw          ❌ FAILED: KeyError: 'in_dim'

Dataset: cic_ids2017_v2
  ▶  rf_5class_smote        

## 6. Summary report

In [ ]:
log_dir = Path(REPO) / 'shap_values'
log_dir.mkdir(parents=True, exist_ok=True)

with open(log_dir / 'notebook_04_run_log.json', 'w') as f:
    json.dump({
        'run_timestamp': datetime.now().isoformat(),
        'total_time_minutes': round(t_total / 60, 1),
        'results': results_log,
        'errors': errors_log,
        'config': {'N_BACKGROUND': N_BACKGROUND, 'N_EVAL': N_EVAL, 'SEED': SEED},
    }, f, indent=2, default=str)

print('Per-model summary:')
print(f'  {"Dataset":<18} {"Model":<22} {"Status":<10} {"Time (s)":<10} {"Shape":<30}')
print('  ' + '-'*92)
for r in results_log:
    t_str = f'{r.get("time_s", ""):.1f}' if r.get('time_s') else ''
    print(f'  {r["dataset"]:<18} {r["model"]:<22} {r["status"]:<10} {t_str:<10} {r.get("shape", ""):<30}')

if errors_log:
    print(f'\n\n❌ FAILURES ({len(errors_log)}):')
    for e in errors_log:
        print(f'  {e["dataset"]}/{e["model"]}: {e["error"]}')

print(f'\nLog saved to: {log_dir / "notebook_04_run_log.json"}')

## 7. Verify outputs

In [ ]:
for dataset in DATASETS:
    print(f'\n{dataset}:')
    out_dir = Path(REPO) / 'shap_values' / dataset
    if not out_dir.exists():
        print('  (no outputs yet)')
        continue
    for model_name in MODELS_5CLASS:
        shap_path = out_dir / f'{model_name}_shap.npy'
        if shap_path.exists():
            sv = np.load(shap_path)
            mean_abs = np.mean(np.abs(sv), axis=(0, 1))
            print(f'  {model_name:<22} shape={sv.shape}, '
                  f'per-class mean|SHAP|: {[f"{v:.4f}" for v in mean_abs]}')
        else:
            print(f'  {model_name:<22} MISSING')

## 8. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/04_shap_analysis.ipynb
!git add shap_values/
!git status --short
!git commit -m 'Notebook 04 v2: SHAP for all 18 5-class models (TreeExplainer + PyTorch GradientExplainer)'
!git push origin main